<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/torneos/notebooks/c6_l1.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C6-L1 · Reglas, eras y submission
12 eras, partición temporal y formato sagrado de submission.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/torneos/data/c6_l1.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c6_l1.csv'), Path('data/c6_l1.csv'), Path('c6_l1.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))


In [ ]:
# Partición por era: el test es estrictamente futuro
train = df[df['era'] <= 8].copy()
test = df[df['era'] > 8].copy()
print(f"train eras {sorted(train['era'].unique())} ({len(train)}) | test eras {sorted(test['era'].unique())} ({len(test)})")
assert len(df) == 60 and df['era'].nunique() == 12
assert train['era'].max() < test['era'].min(), 'el test debe ser posterior al train'


In [ ]:
# Submission desde feature_a: ranking porcentual -> [0, 1]
sub = pd.DataFrame({'id': df['id'], 'prediction': df['feature_a'].rank(pct=True)})
print(sub.describe().to_string())
assert sub['prediction'].between(0, 1).all(), 'predicción fuera de [0, 1]'
assert sub['id'].is_unique, 'ids duplicados'
assert sub.notna().all().all(), 'hay nulos'
print('Formato válido:', sub.shape)


In [ ]:
# Lo prohibido: shuffle global mezcla eras (lookahead encubierto)
mal = df.sample(frac=1.0, random_state=1).reset_index(drop=True)
print('Shuffle mezcla eras en train:', sorted(mal.iloc[:40]['era'].unique())[:6], '...')
assert mal.iloc[:40]['era'].max() > 8, 'el shuffle contamina: ve futuro en train'


In [ ]:
# Chequeo automático L1
assert train['era'].max() < test['era'].min()
assert sub['prediction'].between(0, 1).all() and sub['id'].is_unique
print('OK L1: partición por era + submission válida verificadas')
